In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import torch
from models.feature_generation import build_feature_bank, extract_encoder, pool_features
from preprocessing.dataset import PipistrelleDataset
import pandas as pd
import pickle
from pathlib import Path
import os
from models.abmil_model import train_abmil_all_data, evaluate_abmil_ood

In [ ]:
import pickle

pickle_filename = 'perch2-bags.pkl'
with open(pickle_filename, 'rb') as pickle_file:
    X_bags = pickle.load(pickle_file)

In [ ]:
X_bags_pooled = pool_features(X_bags, windows  = False,window_pooled  = False, method  ='mean',encoder  = 'perch2')

In [ ]:
dir = Path(os.getcwd()).resolve().parent 
path = str(dir / "models" / "features")
y = np.load(path + "\\Y_labels2_not_normalized.npy")

In [ ]:
clf = train_abmil_all_data(X_bags_pooled, y, random_state=42)

In [ ]:
#extract ood perch 2 embeddings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = "perch2"
encoder_model = extract_encoder(encoder)
dir = Path(os.getcwd()).resolve().parent / "data"
batdata = PipistrelleDataset(data_input=str(dir / "ood_metadata.csv"),
                             root_dir=str(dir / "domain-transfer-dataset"),
                             encoder = encoder,
                             filter_echo = False)

X_bags_ood,labels = build_feature_bank(batdata, encoder_model, encoder, device=device)

In [ ]:
pickle_filename = 'perch2-ood-bags.pkl'
with open(pickle_filename, 'wb') as pickle_file:
    pickle.dump(X_bags_ood, pickle_file)


In [ ]:
pickle_filename = 'perch2-ood-label.pkl'
with open(pickle_filename, 'wb') as pickle_file:
    pickle.dump(labels, pickle_file)

In [ ]:
X_bags_pooled_ood = pool_features(X_bags_ood, windows=False, window_pooled=False, method='mean',encoder="perch2")

In [ ]:
y_pred_proba_ood, attention_weights_ood = evaluate_abmil_ood(clf,X_bags_pooled_ood) 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score
from evaluation.tables import generate_master_species_table_with_totals, convert_to_latex_table

# 1. Generate DataFrame with individual species and the bottom global totals row
master_evaluation_table = generate_master_species_table_with_totals(y_pred_proba_ood, str(dir / "ood_metadata.csv"))

# 2. Print Markdown check to terminal
print(master_evaluation_table.to_markdown(index=False))

# 3. Convert and output your final LaTeX code block
latex_code_string = convert_to_latex_table(master_evaluation_table)
print("\n" + "="*40 + " GENERATED LATEX CODE " + "="*40 + "\n")
print(latex_code_string)

# 3. Or save it directly to a .tex snippet file to import into your main document
with open("master_species_table.tex", "w") as f:
    f.write(latex_code_string)